# Sunuwar TTS — Phase 6 prototype fine-tune (Colab T4)

Fine-tunes **`facebook/mms-tts-mai`** (Maithili VITS) on the Sunuwar
single-speaker dataset built by `src/build_tts_dataset.py`.

### Read this before running

**Dataset is 0.88 hours** (472 clips, 12 aligned chapters). Enough to prove the
pipeline end to end and to transfer prosody; **not** enough for an intelligible
voice — single-speaker VITS normally wants 5–20h. The full corpus is 29.5h and
gets there once Phase 3 alignment has run on all 260 chapters. The point of this
run is to retire integration risk while iteration is cheap.

**Why Maithili and not Nepali.** `facebook/mms-tts-nep` does not exist — nor
does `-npi`, nor any Nepali code under `facebook/mms-tts` `full_models/`. Among
the checkpoints that do exist, Maithili covers **98.65%** of this dataset's
characters (hin 97.83%, mar 93.03%, non-Devanagari controls 15.51%), missing only
the danda U+0964. It is also a Nepal contact language. ZWJ is in the vocab, so no
ZWJ remapping is needed.

**Runtime → Change runtime type → T4 GPU** before you start. Then run cells in
order; there is exactly **one** deliberate restart, at step 2.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import sys; print(sys.version)

## 2. Install everything, then restart once

`finetune-hf-vits` is written against transformers 4.x. Current Colab ships
transformers 5.x, where `PretrainedConfig.__getattribute__` raises on undeclared
keys instead of returning `None` — the conversion dies on `config.pad_token_id`.
Pinning the 4.x stack is the fix.

pip will print a wall of red about `cupy`, `jax`, `opencv`, `gradio`, `shap`
etc. wanting numpy≥2. **Ignore it** — none of those are in the VITS path. It is
reporting conflicts among Colab's preinstalled packages, not a failed install.

In [ ]:
%cd /content
![ -d finetune-hf-vits ] || git clone -q https://github.com/ylacombe/finetune-hf-vits.git

!pip install -q -r /content/finetune-hf-vits/requirements.txt

# Pin last, so it wins over whatever requirements.txt pulled in.
!pip install -q "transformers==4.44.2" "huggingface_hub==0.24.6" \
               "datasets==2.21.0" "accelerate==0.34.2" \
               "tokenizers==0.19.1" "numpy<2"

print('\n' + '=' * 70)
print('NOW: Runtime -> Restart session.  Then continue at step 3.')
print('Do NOT use "Disconnect and delete runtime" — that wipes /content.')
print('=' * 70)

## 3. Verify the pinned stack (after the restart)

A restart keeps `/content` — only the Python process resets and Drive unmounts.
So the clone above survives; nothing needs reinstalling.

`pad_token_id present: True` is the line that confirms the original crash is gone.

In [ ]:
import numpy, torch, transformers, datasets, accelerate, huggingface_hub
for m in (numpy, torch, transformers, datasets, accelerate, huggingface_hub):
    print(f'{m.__name__:<18} {m.__version__}')
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

from transformers import VitsConfig
print('pad_token_id present:', hasattr(VitsConfig(), 'pad_token_id'))

assert transformers.__version__.startswith('4.44'), 'restart the session — still on the old process'
assert numpy.__version__.startswith('1.'), 'restart the session — numpy not downgraded yet'
print('\nstack OK')

## 4. Build monotonic_align

VITS' alignment search is a Cython extension that has to be compiled — training
fails at step 1 without it. Built **after** the numpy pin so it links against the
numpy actually in use. It is only importable from the repo directory, which is
where `train_tts.py` launches the trainer from.

In [ ]:
%cd /content/finetune-hf-vits/monotonic_align
!rm -rf build monotonic_align/*.so
!mkdir -p monotonic_align
!python setup.py build_ext --inplace 2>&1 | tail -3

%cd /content/finetune-hf-vits
import monotonic_align
print('monotonic_align OK:', monotonic_align.__file__)

## 5. Mount Drive

Checkpoints go here, so a session timeout costs minutes rather than the run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/sunuwar_tts', exist_ok=True)

## 6. Unpack the dataset

Needs `MyDrive/sunuwar_tts/tts_dataset.zip` (76 MB), built locally with:

```python
import os, zipfile
root = 'data/processed/tts_dataset'
with zipfile.ZipFile('tts_dataset.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for dp, _, fs in os.walk(root):
        for f in fs:
            full = os.path.join(dp, f)
            z.write(full, os.path.relpath(full, root).replace(os.sep, '/'))
```

Use that, not PowerShell's `Compress-Archive` — it writes backslash separators
that `unzip` turns into literal filenames.

The audio is CC BY-NC-ND. Keep it in your own Drive; do not push it to a public
Hub dataset.

In [ ]:
!rm -rf /content/tts_dataset && mkdir -p /content/tts_dataset
!unzip -q /content/drive/MyDrive/sunuwar_tts/tts_dataset.zip -d /content/tts_dataset

import csv, glob
for split in ('train', 'validation'):
    rows = list(csv.DictReader(open(f'/content/tts_dataset/{split}/metadata.csv', encoding='utf-8')))
    wavs = glob.glob(f'/content/tts_dataset/{split}/*.wav')
    print(f'{split:<11} {len(rows):>4} rows, {len(wavs):>4} wavs')
    assert len(rows) == len(wavs) > 0, 'metadata/wav mismatch — re-zip with the snippet above'
print('\ndataset OK')

## 7. Get the project code

Private repo? Use `https://<token>@github.com/...` with a PAT.

In [ ]:
REPO_URL = 'https://github.com/AdityaMallaThakuri/lost-voices-sn.git'
BRANCH = 'week5-phase4-5-tts-prototype'

import os
if not os.path.isdir('/content/Lost-Voices'):
    !git clone -q -b $BRANCH $REPO_URL /content/Lost-Voices

assert os.path.isfile('/content/Lost-Voices/src/train_tts.py'), 'clone failed — upload src/ and configs/ by hand'
!grep -E 'base_checkpoint|discriminator_checkpoint|dataset_dir|output_dir' /content/Lost-Voices/configs/tts.yaml

## 8. Repair the base checkpoint's weight-norm keys

**Do not skip this.** torch ≥2.1 moved `weight_norm` to the parametrization API,
so it expects `parametrizations.weight.original0/original1` where the 2023-era
MMS checkpoint stores `weight_g`/`weight_v`. `from_pretrained` does not error on
the mismatch — it **silently discards those tensors and random-initialises them**.
That is the whole WaveNet stack inside the normalizing flow and the posterior
encoder, i.e. most of what we are trying to transfer.

It is a pure rename, so remapping recovers everything. Expect
`mapped 762, unmatched 0, uncovered 0`.

(The load warnings printed *inside* this cell are expected — it loads the
**original** checkpoint on purpose, just to read its key names.)

In [ ]:
import os, shutil
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file
from transformers import VitsModel

BASE = 'facebook/mms-tts-mai'
REMAP = '/content/mms-tts-mai-remap'

src = snapshot_download(BASE)
shutil.rmtree(REMAP, ignore_errors=True)
shutil.copytree(src, REMAP)

# Stale duplicate with the old key names — remove so it can never be picked.
if os.path.exists(f'{REMAP}/pytorch_model.bin'):
    os.remove(f'{REMAP}/pytorch_model.bin')

sd = load_file(f'{src}/model.safetensors')
target = set(VitsModel.from_pretrained(src).state_dict().keys())

remap, unmatched = {}, []
for k, v in sd.items():
    if k in target:
        remap[k] = v
        continue
    cand = None
    if k.endswith('.weight_g'):
        cand = k[:-9] + '.parametrizations.weight.original0'
    elif k.endswith('.weight_v'):
        cand = k[:-9] + '.parametrizations.weight.original1'
    if cand and cand in target:
        remap[cand] = v
    else:
        unmatched.append(k)

uncovered = sorted(target - set(remap))
print(f'\nsource {len(sd)} keys -> mapped {len(remap)}, unmatched {len(unmatched)}')
print(f'target keys uncovered: {len(uncovered)} {uncovered[:5]}')
assert not unmatched and not uncovered, 'key schema drifted further than the weight_norm rename'

# metadata={'format': 'pt'} is required: transformers 4.44 does
# metadata.get('format') with no None check, and save_file writes
# no header by default.
save_file(remap, f'{REMAP}/model.safetensors', metadata={'format': 'pt'})
print('remapped checkpoint written to', REMAP)

In [ ]:
# Reload the repaired copy. There must be NO "newly initialized" block below —
# only the harmless weight_norm FutureWarning.
m = VitsModel.from_pretrained(REMAP)
print('reloaded cleanly:', round(sum(p.numel() for p in m.parameters()) / 1e6, 1), 'M params')
del m

## 9. Attach the discriminator

The published checkpoint is generator-only, but VITS trains as a GAN. This pulls
Maithili's original `D_100000.pth` and writes a combined checkpoint.

We pass the discriminator and generator paths **separately** rather than using
`--language_code`, because that flag would re-derive the generator as
`facebook/mms-tts-mai` and undo the repair from step 8.

In [ ]:
from huggingface_hub import hf_hub_download

DISC = hf_hub_download('facebook/mms-tts', 'D_100000.pth', subfolder='full_models/mai')
print('discriminator:', DISC)

%cd /content/finetune-hf-vits
!rm -rf /content/mms-tts-base-train
!python convert_original_discriminator_checkpoint.py \
    --checkpoint_path "{DISC}" \
    --generator_checkpoint_path {REMAP} \
    --pytorch_dump_folder_path /content/mms-tts-base-train

!ls -la /content/mms-tts-base-train

Check the output above for a `newly initialized` block. Names under
`discriminator.*` are fine — those come from the separate `D_100000.pth`.
Anything under `flow.*` or `posterior_encoder.*` means step 8 did not take.

## 10. Tokenizer preflight

MMS tokenizers are character-based, so a character the checkpoint has never seen
becomes `<unk>` silently. Expect exactly one missing: **U+0964 (danda), 472
occurrences** — sentence-final punctuation the narrator does not voice, one per
clip. It gets dropped.

Anything else in the missing list is worth stopping for.

In [ ]:
%cd /content/Lost-Voices
!python src/train_tts.py configs/tts.yaml --preflight

## 11. Config compatibility check

`run_vits_finetuning.py` changes over time. Diff our keys against the field names
in its source before committing to a two-hour run. A *renamed* key means our
setting is silently ignored.

In [ ]:
import re, json, sys, yaml
from pathlib import Path
sys.path.insert(0, '/content/Lost-Voices/src')
from train_tts import build_training_json

cfg = yaml.safe_load(Path('/content/Lost-Voices/configs/tts.yaml').read_text(encoding='utf-8'))
ours = build_training_json(cfg, Path('/content/tts_dataset'), Path('/content/_tts_finetune.json'))

src_txt = Path('/content/finetune-hf-vits/run_vits_finetuning.py').read_text(encoding='utf-8')
known = set(re.findall(r'^\s{4}(\w+)\s*:\s*\w', src_txt, flags=re.M)) | set(re.findall(r'--([\w-]+)', src_txt))

unknown = sorted(k for k in ours if k not in known)
print(f'{len(ours)} keys ours / {len(known)} found in trainer')
print('NOT FOUND:', unknown or 'none')

In [ ]:
# ONLY if the check above flagged data_dir: materialise the dataset to disk.
# Keeps the audio local — no upload anywhere.
from datasets import load_dataset, Audio
ds = load_dataset('audiofolder', data_dir='/content/tts_dataset')
ds = ds.cast_column('audio', Audio(sampling_rate=16000))
print(ds)
ds.save_to_disk('/content/tts_dataset_hf')

## 12. Train

362 clips at batch 8 ≈ 45 steps/epoch; 200 epochs ≈ 9k steps, roughly 1.5–2.5h on
a T4. Checkpoints land in Drive every 250 steps.

Watch the mel loss. Plateauing early with buzzy audio is the 0.88h dataset
talking, not a bug. Note the wall-clock time — it scales the full-corpus estimate.

### Patch the trainer's single-speaker path

`speaker_id_column_name` is null (one narrator), so the collator never puts a
`speaker_id` in the batch — but the training loop reads `batch["speaker_id"]`
unconditionally and dies with `KeyError: 'speaker_id'` on the first step. The
model accepts `None` here because `speaker_embedding_size` is 0.

Only the three **call-site reads** change. Lines 380 and 753 are dict
*assignments* and must stay as they are — matching on the `speaker_id=` prefix
is what keeps them untouched.

In [ ]:
from pathlib import Path

p = Path('/content/finetune-hf-vits/run_vits_finetuning.py')
s = p.read_text()
n = s.count('speaker_id=batch["speaker_id"]')
s = s.replace('speaker_id=batch["speaker_id"]', 'speaker_id=batch.get("speaker_id")')
p.write_text(s)
print('patched', n, 'reads (expect 3, or 0 if already applied)')

!python -m py_compile /content/finetune-hf-vits/run_vits_finetuning.py && echo 'COMPILES OK'
!grep -n 'speaker_id"\]' /content/finetune-hf-vits/run_vits_finetuning.py

### Restore `tostring_rgb` for matplotlib >= 3.8

The trainer plots the attention alignment during validation via
`utils/plot.py`, which calls `FigureCanvasAgg.tostring_rgb()` — removed in
matplotlib 3.8. Training runs fine until the first eval (step 50 by default)
and then dies, so this must be applied before starting.

`buffer_rgba()` is the replacement but has 4 channels while the callers
reshape to `(h, w, 3)`, so the shim drops alpha and returns exactly the old
bytes. Appended, not prepended: `plot.py` sets the matplotlib backend at the
top and an import above that could force a backend too early.

In [ ]:
from pathlib import Path

p = Path('/content/finetune-hf-vits/utils/plot.py')
s = p.read_text()

shim = '''

# matplotlib >= 3.8 removed FigureCanvasAgg.tostring_rgb().
import matplotlib.backends.backend_agg as _agg
import numpy as _np

if not hasattr(_agg.FigureCanvasAgg, "tostring_rgb"):
    def _tostring_rgb(self):
        return _np.asarray(self.buffer_rgba())[..., :3].tobytes()
    _agg.FigureCanvasAgg.tostring_rgb = _tostring_rgb
'''

if 'tostring_rgb = _tostring_rgb' not in s:
    p.write_text(s + shim)
    print('shim appended')
else:
    print('already patched')

import sys; sys.path.insert(0, '/content/finetune-hf-vits')
import utils.plot, matplotlib.backends.backend_agg as _a
print('tostring_rgb restored:', hasattr(_a.FigureCanvasAgg, 'tostring_rgb'))

In [ ]:
%cd /content/Lost-Voices
!python src/train_tts.py configs/tts.yaml

## 13. Synthesise — the demo

Held-out **validation** sentences (chapters LUK_005 and REV_021, never seen in
training), with the real recording next to the synthesised one.

In [ ]:
import csv, glob, torch, scipy.io.wavfile
from pathlib import Path
from IPython.display import Audio as PlayAudio, display
from transformers import VitsModel, AutoTokenizer

OUT = '/content/drive/MyDrive/sunuwar_tts/prototype'
ckpts = sorted(glob.glob(f'{OUT}/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1]))
CKPT = ckpts[-1] if ckpts else OUT
print('loading', CKPT)

model = VitsModel.from_pretrained(CKPT).eval()
tokenizer = AutoTokenizer.from_pretrained(CKPT)

rows = list(csv.DictReader(open('/content/tts_dataset/validation/metadata.csv', encoding='utf-8')))
samples = Path('/content/drive/MyDrive/sunuwar_tts/samples'); samples.mkdir(parents=True, exist_ok=True)

for row in rows[:5]:
    inputs = tokenizer(row['text'], return_tensors='pt')
    with torch.no_grad():
        wav = model(**inputs).waveform[0].cpu().numpy()
    name = Path(row['file_name']).stem
    scipy.io.wavfile.write(samples / f'{name}_synth.wav', model.config.sampling_rate, wav)

    print('=' * 78)
    print(row['text'])
    print('reference:');   display(PlayAudio(f"/content/tts_dataset/validation/{row['file_name']}"))
    print('synthesised:'); display(PlayAudio(wav, rate=model.config.sampling_rate))

## 14. Free-text synthesis

The same adaptation the training text went through has to be applied here, or the
tokenizer sees characters the model never trained on.

In [ ]:
from train_tts import build_policy, adapt_text

TEXT = 'परमप्रभु यावे आ दाक्शो पा'   # <-- your sentence

vocab = set(tokenizer.get_vocab())
missing = {c: 1 for c in set(TEXT) if c not in vocab and not c.isspace()}
clean = adapt_text(TEXT, missing, build_policy(missing, cfg)) if missing else TEXT
if missing:
    print('adapted:', [f'U+{ord(c):04X}' for c in missing], '->', clean)

inputs = tokenizer(clean, return_tensors='pt')
with torch.no_grad():
    wav = model(**inputs).waveform[0].cpu().numpy()
display(PlayAudio(wav, rate=model.config.sampling_rate))

## What to bring back

These decide the Phase 3 settings:

1. **Preflight output (step 10)** — confirm the danda was the only missing char.
2. **Final train/eval loss**, and whether it was still descending at the end.
3. **How it sounds** — Sunuwar-*sounding* nonsense (prosody transferred,
   phonetics not learned yet, expected at 0.88h) versus pure noise (a real bug).
4. **Wall-clock for 200 epochs** — scales the full-corpus estimate.

Checkpoints are in Drive at `MyDrive/sunuwar_tts/prototype`, samples in
`MyDrive/sunuwar_tts/samples`. Do not try to commit `model.safetensors` to
GitHub — it exceeds the 100 MB file limit and is gitignored.